# 🎵 KTV 歌詞對齊（Qwen3-ForcedAligner）

**用你自己的歌詞去對齊歌曲，輸出每個字的時間。**

### 使用方法
1. 上方選單：**執行階段 → 變更執行階段類型 → 選 T4 GPU**（沒有 GPU 也能跑，但很慢）。
2. 按下面第 1 個儲存格左邊的 ▶ 執行（安裝，約 3～5 分鐘，每次開新的 Colab 都要做）。
   - 如果跳出「必須重新啟動工作階段」，按重新啟動，之後**不用再跑第 1 格**，直接跑第 2 格。
3. 按第 2 個儲存格的 ▶，依畫面提示上傳**歌曲**和**歌詞 .txt**（可以一次選兩個檔案）。
4. 完成後會自動下載一個 zip，裡面有：`_enhanced.lrc`（逐字）、`.lrc`（整句）、`.srt`、`.ass`（KTV 卡拉OK 效果）。

### 注意
- 歌曲請在 **5 分鐘以內**（這個模型的上限），最好是**純人聲**。
- 歌詞 **.txt 一行一句**，不要放歌名、歌手。
- 時間是模型估計的，不保證每個字都準。如果整體固定偏早或偏晚，用 `offset_ms` 修正。


In [ ]:
#@title 1. 安裝（每次開新的 Colab 都要先按這個，約 3～5 分鐘）
!pip install -q -U qwen-asr
import torch
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 沒有 GPU。請到 執行階段 → 變更執行階段類型 選 T4 GPU；沒有的話會很慢。")


In [ ]:
#@title 2. 上傳歌曲和歌詞，開始對齊
language = "Chinese" #@param ["Chinese", "English", "Cantonese", "Japanese", "Korean", "French", "German", "Spanish", "Italian", "Portuguese", "Russian"]
separate_vocals = False #@param {type:"boolean"}
skip_bracket_lines = True #@param {type:"boolean"}
offset_ms = 0 #@param {type:"integer"}
output_name = "lyrics" #@param {type:"string"}

import os, re, sys, bisect, shutil, subprocess, unicodedata
from collections import namedtuple

# ===== helpers =====
CJK_LANGS = ("Chinese", "Cantonese", "Japanese", "Korean")
CJK_RANGE = "\u3040-\u30ff\u3400-\u9fff\uf900-\ufaff\uac00-\ud7af"
TIME_TAG = re.compile(r"\[\d{1,3}:\d{2}(?:[.:]\d{1,3})?\]|<\d{1,3}:\d{2}(?:[.:]\d{1,3})?>")
BRACKET_LINE = re.compile(r"^\s*[\(\[（【\{「『<].*[\)\]）】\}」』>]\s*$")
Unit = namedtuple("Unit", "text start end")

def read_text_file(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8-sig", "utf-8", "big5", "gb18030"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="ignore")

def is_cjk(ch):
    return re.match("[" + CJK_RANGE + "]", ch) is not None

def clean_lyrics(text, language, skip_brackets=True):
    lines = []
    for line in text.replace("\r", "").split("\n"):
        line = TIME_TAG.sub("", line).strip()
        if not line:
            continue
        if skip_brackets and BRACKET_LINE.match(line):
            continue
        # punctuation -> space (keep ' and - inside words)
        chars = []
        for c in line:
            if unicodedata.category(c).startswith("P") and c not in "'-":
                chars.append(" ")
            else:
                chars.append(c)
        line = re.sub(r"\s+", " ", "".join(chars)).strip()
        if language in CJK_LANGS:
            line = re.sub("(?<=[" + CJK_RANGE + r"])\s+(?=[" + CJK_RANGE + "])", "", line)
        if line:
            lines.append(line)
    return lines

def units_from_results(results, audio_seconds):
    seq = results[0]
    if not hasattr(seq, "__iter__"):
        seq = seq.items
    units = [Unit(str(it.text), float(it.start_time), float(it.end_time)) for it in seq]
    if units and max(u.end for u in units) > audio_seconds * 1.5 + 5:
        units = [u._replace(start=u.start / 1000.0, end=u.end / 1000.0) for u in units]  # milliseconds
    return units

def assign_units_to_lines(lines, units):
    full = "\n".join(lines)
    starts, pos = [], 0
    for l in lines:
        starts.append(pos)
        pos += len(l) + 1
    per_line = [[] for _ in lines]
    cursor, skipped = 0, 0
    for u in units:
        t = u.text.strip()
        if not t:
            continue
        idx = full.find(t, cursor)
        if idx < 0 or idx - cursor > 12:
            skipped += 1
            continue
        per_line[bisect.bisect_right(starts, idx) - 1].append(u._replace(text=t))
        cursor = idx + len(t)
    return per_line, skipped

def apply_offset(per_line, offset_s):
    out = []
    for units in per_line:
        out.append([u._replace(start=max(0.0, u.start + offset_s), end=max(0.0, u.end + offset_s)) for u in units])
    return out

def needs_space(a, b):
    if not a or not b or is_cjk(a[-1]) or is_cjk(b[0]):
        return False
    return a[-1].isalnum() and b[0].isalnum()

def join_units(units):
    text = ""
    for i, u in enumerate(units):
        text += u.text
        if i + 1 < len(units) and needs_space(u.text, units[i + 1].text):
            text += " "
    return text

def fmt_lrc(t):
    cs = int(round(t * 100)); return "%02d:%02d.%02d" % (cs // 6000, (cs % 6000) // 100, cs % 100)

def fmt_srt(t):
    ms = int(round(t * 1000))
    return "%02d:%02d:%02d,%03d" % (ms // 3600000, (ms % 3600000) // 60000, (ms % 60000) // 1000, ms % 1000)

def fmt_ass(t):
    cs = int(round(t * 100))
    return "%d:%02d:%02d.%02d" % (cs // 360000, (cs % 360000) // 6000, (cs % 6000) // 100, cs % 100)

def write_enhanced_lrc(path, per_line):
    with open(path, "w", encoding="utf-8") as f:
        f.write("[re:KTV aligner]\n")
        for units in per_line:
            if not units:
                continue
            line = "[" + fmt_lrc(units[0].start) + "]"
            for i, u in enumerate(units):
                line += "<" + fmt_lrc(u.start) + ">" + u.text
                if i + 1 < len(units) and needs_space(u.text, units[i + 1].text):
                    line += " "
            line += "<" + fmt_lrc(units[-1].end) + ">"
            f.write(line + "\n")

def write_line_lrc(path, per_line):
    with open(path, "w", encoding="utf-8") as f:
        for units in per_line:
            if units:
                f.write("[" + fmt_lrc(units[0].start) + "]" + join_units(units) + "\n")

def write_srt(path, per_line):
    n = 0
    with open(path, "w", encoding="utf-8") as f:
        for units in per_line:
            if not units:
                continue
            n += 1
            start, end = units[0].start, max(units[-1].end, units[0].start + 0.3)
            f.write("%d\n%s --> %s\n%s\n\n" % (n, fmt_srt(start), fmt_srt(end), join_units(units)))

def escape_ass(text):
    return text.replace("\\", "").replace("{", "(").replace("}", ")")

def write_ass(path, per_line):
    with open(path, "w", encoding="utf-8") as f:
        f.write("[Script Info]\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nPlayResX: 1920\nPlayResY: 1080\n\n")
        f.write("[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n")
        f.write("Style: Karaoke,Arial,72,&H0000FFFF,&H00FFFFFF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,4,0,2,80,80,110,1\n\n")
        f.write("[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n")
        lines = [u for u in per_line if u]
        prev_end = 0.0
        for i, units in enumerate(lines):
            first, last = units[0].start, units[-1].end
            if last <= first:
                continue
            nxt = lines[i + 1][0].start if i + 1 < len(lines) else float("inf")
            ds = min(max(prev_end, max(0.0, first - 0.4)), first)
            de = max(last, min(last + 0.5, nxt))
            prev_end = de
            text, cursor = "", 0
            for j, u in enumerate(units):
                s = int(round((u.start - ds) * 100)); e = int(round((u.end - ds) * 100))
                s = max(s, cursor)
                if s > cursor:
                    text += "{\\k%d}" % (s - cursor)
                d = max(1, e - s)
                text += "{\\kf%d}%s" % (d, escape_ass(u.text))
                if j + 1 < len(units) and needs_space(u.text, units[j + 1].text):
                    text += " "
                cursor = s + d
            f.write("Dialogue: 0,%s,%s,Karaoke,,0,0,0,,%s\n" % (fmt_ass(ds), fmt_ass(de), text))
# ===== end helpers =====

if __name__ == "__main__" or "google.colab" in sys.modules:
    from google.colab import files
    import torch

    # ---- 1. upload ----
    AUDIO_EXT = (".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".mp4", ".mkv", ".mov", ".webm")
    audio_path = lyrics_path = None
    for round_no in range(3):
        if audio_path and lyrics_path:
            break
        need = []
        if not audio_path: need.append("歌曲音檔")
        if not lyrics_path: need.append("歌詞 .txt")
        print("請選擇：" + "、".join(need) + "（可以一次選多個檔案）")
        for name in files.upload():
            low = name.lower()
            if low.endswith(".txt"):
                lyrics_path = name
            elif low.endswith(AUDIO_EXT):
                audio_path = name
    if not (audio_path and lyrics_path):
        raise SystemExit("❌ 需要一個音檔和一個 .txt 歌詞檔。請重新執行這個儲存格。")

    # ---- 2. lyrics ----
    lines = clean_lyrics(read_text_file(lyrics_path), language, skip_bracket_lines)
    if not lines:
        raise SystemExit("❌ 歌詞檔是空的，或所有行都被當成標記略過了。")
    print("歌詞共 %d 行，第一行：%s" % (len(lines), lines[0]))

    # ---- 3. audio ----
    def duration_of(path):
        out = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", path],
                             capture_output=True, text=True).stdout.strip()
        return float(out)

    source = audio_path
    if separate_vocals:
        try:
            print("分離人聲中（會比較久）…")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "demucs"], check=True)
            subprocess.run([sys.executable, "-m", "demucs", "--two-stems=vocals", "-n", "htdemucs", "-o", "sep", audio_path], check=True)
            stem = os.path.splitext(os.path.basename(audio_path))[0]
            candidate = os.path.join("sep", "htdemucs", stem, "vocals.wav")
            if os.path.exists(candidate):
                source = candidate
                print("✅ 已使用分離後的人聲")
            else:
                print("⚠️ 找不到分離後的人聲，改用原始音檔")
        except Exception as e:
            print("⚠️ 人聲分離失敗，改用原始音檔：", e)

    subprocess.run(["ffmpeg", "-y", "-i", source, "-ac", "1", "-ar", "16000", "work.wav"], check=True, capture_output=True)
    seconds = duration_of("work.wav")
    print("音檔長度：%.1f 秒" % seconds)
    if seconds > 300:
        raise SystemExit("❌ 音檔超過 5 分鐘（%.0f 秒），這個模型最多 5 分鐘。請先把歌曲和歌詞切成 5 分鐘內的段落。" % seconds)

    # ---- 4. align ----
    from qwen_asr import Qwen3ForcedAligner
    if torch.cuda.is_available():
        dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        device = "cuda:0"
    else:
        dtype, device = torch.float32, "cpu"
        print("⚠️ 使用 CPU，會比較慢")
    print("載入對齊模型…")
    model = Qwen3ForcedAligner.from_pretrained("Qwen/Qwen3-ForcedAligner-0.6B", dtype=dtype, device_map=device)
    joiner = "" if language in CJK_LANGS else " "
    print("對齊中…")
    results = model.align(audio="work.wav", text=joiner.join(lines), language=language)

    units = units_from_results(results, seconds)
    per_line, skipped = assign_units_to_lines(lines, units)
    per_line = apply_offset(per_line, offset_ms / 1000.0)
    aligned_lines = sum(1 for u in per_line if u)
    print("對齊完成：%d 個字詞，%d / %d 行有時間，%d 個字詞沒對上" % (len(units), aligned_lines, len(lines), skipped))

    # ---- 5. write ----
    name = re.sub(r"[^\w\-]+", "_", output_name).strip("_") or "lyrics"
    outputs = [name + "_enhanced.lrc", name + ".lrc", name + ".srt", name + ".ass"]
    write_enhanced_lrc(outputs[0], per_line)
    write_line_lrc(outputs[1], per_line)
    write_srt(outputs[2], per_line)
    write_ass(outputs[3], per_line)

    print("\n--- 預覽（前 10 行）---")
    shown = 0
    for line, units in zip(lines, per_line):
        if shown >= 10:
            break
        if units:
            print("[%s -> %s] %s" % (fmt_lrc(units[0].start), fmt_lrc(units[-1].end), line))
        else:
            print("[  沒有時間  ] %s" % line)
        shown += 1

    import zipfile
    zip_name = name + "_result.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for fn in outputs:
            z.write(fn)
    files.download(zip_name)
    print("\n✅ 完成，zip 已開始下載。")
